# Newsvendor Game -- Module 2: What Actually Drives Q*?

**This is a take-home module.** One product -- the Downtown Boutique sneaker
drop -- run through a series of short scenarios. Each scenario changes
exactly **one lever** behind the newsvendor formula, so you can see, in
isolation, what that lever does.

## The levers, in order

| # | Lever | The story |
|---|---|---|
| 0 | Baseline | Play the drop cold, estimate mu/sigma yourself |
| 1 | **Co (overage cost)** | Storage rent hikes, salvage payout drops |
| 2 | **Cu (underage cost)** | A rival closes, you raise price |
| 3 | **Cu and Co together, ratio unchanged** | Across-the-board inflation |
| 4 | **Demand mean (mu)** | A blogger post goes viral |
| 5 | **Demand uncertainty (sigma)** | You launch in an unfamiliar city |
| 6 | **Service level, no cost given** | Corporate just sets a 90% in-stock target |
| 7 | **Sensitivity curves** | How Q* responds continuously to Cu and Co |
| 8 | **Behavioral framing** | Same numbers, described two different ways |
| 9 | Whole-story summary | One chart, everything side by side |

**This notebook is a menu, not a checklist.** For your 8-minute, 8-slide
presentation, **Chapters 0, 4, 6, and 9 are required** -- plus **any 2 more**
of your choosing from Chapters 1, 2, 3, 5, 7, or 8. Beyond that, add more
chapters only if you have time left -- see the presentation guide at the end.

---

## Deliverables & In-Class Presentation Guide (8 minutes)

### Required
Submit CSVs for every chapter you played, plus a table: for each chapter,
your average order, mu-hat, sigma-hat, critical ratio, Q*, and delta-q.

### Slide plan -- ~ 8 minutes (~1 minute each slide)

**Required (4 slides) -- always present these:**
- **Chapter 0** -- baseline delta-q
- **Chapter 4** -- demand-mean shock: did Q* and your order move together?
- **Chapter 6** -- service level: how a policy target compares to the
  margin-based critical ratio
- **Chapter 9** -- the whole-story chart, as your closing slide
- **Choose any 2 more (2 slides)** from Chapters 1, 2, 3, 5, 7, or 8 -- pick
whichever taught you the most or surprised you.

**Optional, time permitting (up to 2 more slides):** add more chapters from
the same list only if you're comfortably inside the 8-minute limit. It's
fine to stop at 6 slides if that's a tighter, cleaner story.

### Grading emphasis
I personally care less about whether your Q* estimate was exactly "correct" and more
about whether you can **explain what each lever did and why**, and whether
you can honestly point out where your own intuition diverged from the
formula.

In [ ]:
# ============================================================
# SHARED LIBRARY -- run this cell first.
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, HTML, clear_output
from statistics import NormalDist
from dataclasses import dataclass, asdict
from typing import Optional, List
import os, datetime

try:
    from google.colab import files
    ON_COLAB = True
except Exception:
    ON_COLAB = False

SESSION_LOG = {}

class NewsvendorEngine:
    def __init__(self, p, c, h, mu, sigma, name="Product", rng=None):
        self.p, self.c, self.h = p, c, h
        self.mu, self.sigma = mu, sigma
        self.name = name
        self.rng = rng or np.random.default_rng()

    def sample_demand(self, n=1):
        draws = np.clip(self.rng.normal(self.mu, self.sigma, size=n), 0, None)
        return draws[0] if n == 1 else draws

    def critical_ratio(self):
        Cu, Co = self.p - self.c, self.c - self.h
        return Cu / (Cu + Co) if (Cu + Co) > 0 else 0.0

    def q_star(self):
        cr = min(max(self.critical_ratio(), 1e-6), 1 - 1e-6)
        z = NormalDist().inv_cdf(cr)
        return max(0.0, self.mu + z * self.sigma)

    def q_star_from_csl(self, csl):
        cr = min(max(csl, 1e-6), 1 - 1e-6)
        z = NormalDist().inv_cdf(cr)
        return max(0.0, self.mu + z * self.sigma)

    def profit(self, q, x):
        profit = (self.p * x - self.c * q + self.h * (q - x)) if x < q else (self.p - self.c) * q
        return profit

    def simulate_policy(self, q, n=3000):
        xs = self.sample_demand(n)
        return np.array([self.profit(q, x) for x in xs])

print("Shared library loaded.")


Shared library loaded.


In [ ]:
# ============================================================
# Single-product session runner, reused for every chapter below.
# ============================================================
@dataclass
class SPRoundResult:
    round_num: int
    order_q: float
    demand_x: float
    profit: float
    cum_profit: float
    running_mean: float
    running_sd: Optional[float]

def build_single_product_widget(engine, rounds=8, order_min=0, order_max=None, order_step=10,
                                 frame_text=None, session_tag="chapter", title="Newsvendor Session",
                                 record_qstar=None):
    order_max = order_max or (engine.mu + 4 * engine.sigma)
    STATE = {"results": [], "cum_profit": 0.0, "current_round": 1, "game_over": False}

    order_in = w.BoundedFloatText(value=round(engine.mu), min=order_min, max=order_max, step=order_step,
                                   description="Order q:")
    start_btn = w.Button(description="Start / Reset", button_style="primary")
    submit_btn = w.Button(description="Submit", disabled=True)
    download_btn = w.Button(description="Download CSV", disabled=True)
    metrics_out, msg_out = w.Output(), w.Output()
    hist_table_out, hist_chart_out = w.Output(), w.Output()
    summary_out, results_out = w.Output(), w.Output()

    def render_metrics():
        metrics_out.clear_output()
        with metrics_out:
            print(f"Round {min(STATE['current_round'], rounds)} / {rounds}   |   Cumulative profit: {STATE['cum_profit']:.2f}")

    def render_history():
        hist_table_out.clear_output(); hist_chart_out.clear_output()
        res = STATE["results"]
        with hist_table_out:
            if not res:
                print("No rounds played yet."); return
            df = pd.DataFrame({
                "Round": [r.round_num for r in res],
                "Demand": [round(r.demand_x, 1) for r in res],
                "Running Mean": [round(r.running_mean, 2) for r in res],
                "Running SD": [round(r.running_sd, 2) if r.running_sd is not None else None for r in res],
            })
            display(df)
            last = res[-1]
            sd_txt = f"{last.running_sd:.2f}" if last.running_sd is not None else "N/A"
            print(f"mu-hat = {last.running_mean:.2f}  |  sigma-hat = {sd_txt}")
        if not res:
            return
        with hist_chart_out:
            fig, ax = plt.subplots(figsize=(6.5, 3))
            ax.plot([r.round_num for r in res], [r.demand_x for r in res], marker='o', label="Demand")
            ax.plot([r.round_num for r in res], [r.running_mean for r in res], linestyle='--', label="Running mean")
            ax.set_xlabel("Round"); ax.set_ylabel("Demand"); ax.legend(); ax.grid(alpha=0.3)
            plt.tight_layout(); plt.show()

    def reset(_=None):
        STATE.update({"results": [], "cum_profit": 0.0, "current_round": 1, "game_over": False})
        order_in.disabled = False; submit_btn.disabled = False; download_btn.disabled = True
        for o in [msg_out, summary_out, results_out]:
            o.clear_output()
        render_metrics(); render_history()
        with msg_out:
            display(HTML(f"<b>{title}</b> started. Enter your order and click Submit."))

    def submit(_=None):
        if STATE["game_over"]:
            return
        q = order_in.value
        x = float(engine.sample_demand(1))
        profit = engine.profit(q, x)
        STATE["cum_profit"] += profit
        prior = [r.demand_x for r in STATE["results"]]
        all_d = prior + [x]
        running_mean = float(np.mean(all_d))
        running_sd = float(np.std(all_d, ddof=1)) if len(all_d) > 1 else None
        rr = SPRoundResult(STATE["current_round"], q, x, profit, STATE["cum_profit"], running_mean, running_sd)
        STATE["results"].append(rr)
        STATE["current_round"] += 1
        if STATE["current_round"] > rounds:
            STATE["game_over"] = True
        render_metrics(); render_history()
        with msg_out:
            clear_output()
            display(HTML(f"Order: <b>{q:.0f}</b> | Demand: <b>{x:.1f}</b> | Profit: <b>{profit:.2f}</b>"))
        if STATE["game_over"]:
            qstar_reported = record_qstar if record_qstar is not None else engine.q_star()
            with summary_out:
                clear_output()
                orders = [r.order_q for r in STATE["results"]]
                print(f"Average order: {np.mean(orders):.2f}")
                print(f"Estimated mu-hat: {STATE['results'][-1].running_mean:.2f}, sigma-hat: {STATE['results'][-1].running_sd:.2f}")
                print(f"Critical ratio: {engine.critical_ratio():.3f}   |   Q*: {qstar_reported:.1f}")
                print(f"Total profit: {STATE['cum_profit']:.2f}")
            with results_out:
                display(pd.DataFrame([asdict(r) for r in STATE["results"]]))
            SESSION_LOG[session_tag] = {
                "avg_order": float(np.mean(orders)),
                "q_star": qstar_reported,
                "critical_ratio": engine.critical_ratio(),
                "mu": engine.mu, "sigma": engine.sigma,
                "p": engine.p, "c": engine.c, "h": engine.h,
                "results": [asdict(r) for r in STATE["results"]],
            }
            download_btn.disabled = False
            order_in.disabled = True; submit_btn.disabled = True

    def download(_=None):
        if not STATE["results"]:
            return
        df = pd.DataFrame([asdict(r) for r in STATE["results"]])
        fname = f"newsvendor_{session_tag}_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.csv"
        path = os.path.join("/content" if ON_COLAB else ".", fname)
        df.to_csv(path, index=False)
        if ON_COLAB:
            files.download(path)
        else:
            print(f"Saved to {path}")

    start_btn.on_click(reset); submit_btn.on_click(submit); download_btn.on_click(download)

    header = w.HTML(f"<h3>{title}</h3>")
    frame_html = w.HTML(f"<div style='background:#f6f6f6;padding:10px;border-radius:6px;'>{frame_text}</div>") if frame_text else w.HTML("")
    box = w.VBox([header, frame_html, w.HBox([order_in]), w.HBox([start_btn, submit_btn, download_btn]),
                  metrics_out, msg_out, hist_table_out, hist_chart_out, summary_out,
                  w.HTML("<h4>Round-by-round results</h4>"), results_out])
    return box, STATE

def compare_bar(labels, q_stars, avg_orders, title):
    x = np.arange(len(labels)); width = 0.35
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.bar(x - width/2, q_stars, width, label="Q*", color="#DD8452")
    ax.bar(x + width/2, avg_orders, width, label="Your average order", color="#4C72B0")
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("Order quantity")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout(); plt.show()


## Chapter 0 -- Baseline: The Downtown Boutique Sneaker Drop

Selling price $180, cost $95, salvage $40. Demand is roughly normal; you are
**not told** mu or sigma -- estimate them the way you did in Module 1.

**What to do:** play 30 rounds, then run the analysis cell.


In [ ]:
baseline_engine = NewsvendorEngine(p=180, c=95, h=40, mu=300, sigma=60, name="Baseline")
box0, STATE0 = build_single_product_widget(baseline_engine, rounds=30, order_min=0, order_max=500,
                                            order_step=5, session_tag="baseline", title="Chapter 0: Baseline")
display(box0)


In [ ]:
if "baseline" in SESSION_LOG:
    s = SESSION_LOG["baseline"]
    compare_bar(["Baseline"], [s["q_star"]], [s["avg_order"]], "Chapter 0: Your order vs. Q*")
    print(f"Critical ratio: {s['critical_ratio']:.3f}")
    print(f"delta-q: {s['avg_order'] - s['q_star']:+.1f}")
else:
    print("Play all 30 rounds of Chapter 0 first.")


**Questions to answer -- Chapter 0:**
1. What is your estimated mu-hat and sigma-hat after 30 rounds?
2. What is the critical ratio for this product? Show Cu, Co, and the calculation.
3. What is the formula's Q*?
4. What was your average order across the 30 rounds?
5. What is delta-q (your average order minus Q*)? Did you tend to over-order
   or under-order relative to the formula?


## Chapter 1 -- Co Shock: Storage Costs Rise, Salvage Falls

*"Your landlord raised warehouse rent, and your outlet partner cut what
they'll pay for leftover pairs: salvage drops from $40 to $10. Price and
cost are unchanged."*

This raises **Co = c - h** (from $55 to $85) while **Cu stays at $85**.

**Predict first:** does the critical ratio go up or down? Should you order
more or less than baseline?

**What to do:** play 30 rounds, then compare to Chapter 0.


In [ ]:
co_shock_engine = NewsvendorEngine(p=180, c=95, h=10, mu=300, sigma=60, name="Co Shock")
box1, STATE1 = build_single_product_widget(co_shock_engine, rounds=30, order_min=0, order_max=500,
                                            order_step=5, session_tag="co_shock", title="Chapter 1: Co Shock")
display(box1)


In [ ]:
if "co_shock" in SESSION_LOG and "baseline" in SESSION_LOG:
    b, s = SESSION_LOG["baseline"], SESSION_LOG["co_shock"]
    compare_bar(["Baseline", "Co shock"], [b["q_star"], s["q_star"]], [b["avg_order"], s["avg_order"]],
                "Chapter 1: Overage cost up -> Q* should fall")
    print(f"Co -- baseline: {b['c']-b['h']:.0f}  ->  shock: {s['c']-s['h']:.0f}")
    print(f"Critical ratio -- baseline: {b['critical_ratio']:.3f}  ->  shock: {s['critical_ratio']:.3f}")
    print(f"Q* -- baseline: {b['q_star']:.1f}  ->  shock: {s['q_star']:.1f}  ({s['q_star']-b['q_star']:+.1f})")
else:
    print("Play both Chapter 0 and Chapter 1 fully before running this cell.")


Play both Chapter 0 and Chapter 1 fully before running this cell.


**Questions to answer -- Chapter 1 (Co shock):**
1. What is the new Co, and how does it compare to baseline Co ($55)?
2. What is the new critical ratio compared to baseline (0.607)?
3. Did your average order fall from Chapter 0 to Chapter 1? By how much?
4. Did the formula's Q* fall by more or less than your own order did?
5. In one sentence: why does a lower salvage value push the optimal order down?


## Chapter 2 -- Cu Shock: A Rival Closes, You Raise Price

*"The boutique across town just shut down. With less competition you raise
your price from $180 to $220. Cost and salvage are unchanged."*

This raises **Cu = p - c** (from $85 to $125) while **Co stays at $55**.

**Predict first:** which direction does Q* move this time?

**What to do:** play 30 rounds, then compare to Chapter 0.


In [ ]:
cu_shock_engine = NewsvendorEngine(p=220, c=95, h=40, mu=300, sigma=60, name="Cu Shock")
box2, STATE2 = build_single_product_widget(cu_shock_engine, rounds=30, order_min=0, order_max=500,
                                            order_step=5, session_tag="cu_shock", title="Chapter 2: Cu Shock")
display(box2)


In [ ]:
if "cu_shock" in SESSION_LOG and "baseline" in SESSION_LOG:
    b, s = SESSION_LOG["baseline"], SESSION_LOG["cu_shock"]
    compare_bar(["Baseline", "Cu shock"], [b["q_star"], s["q_star"]], [b["avg_order"], s["avg_order"]],
                "Chapter 2: Underage cost up -> Q* should rise")
    print(f"Cu -- baseline: {b['p']-b['c']:.0f}  ->  shock: {s['p']-s['c']:.0f}")
    print(f"Critical ratio -- baseline: {b['critical_ratio']:.3f}  ->  shock: {s['critical_ratio']:.3f}")
    print(f"Q* -- baseline: {b['q_star']:.1f}  ->  shock: {s['q_star']:.1f}  ({s['q_star']-b['q_star']:+.1f})")
else:
    print("Play both Chapter 0 and Chapter 2 fully before running this cell.")


**Questions to answer -- Chapter 2 (Cu shock):**
1. What is the new Cu, and how does it compare to baseline Cu ($85)?
2. What is the new critical ratio?
3. Did your average order rise from Chapter 0 to Chapter 2? By how much?
4. Compare the size of your order change here to the size of your change in
   Chapter 1 -- which shock moved your gut order more, the Co shock or the
   Cu shock? Give a possible reason why.


## Chapter 3 -- Cu AND Co Both Change, but the Ratio Doesn't

*"Nationwide inflation hits everyone at once: your supplier raises cost by
20%, you pass that fully to customers by raising retail price 20%, and your
outlet partner adjusts their payout by the same 20%."*

p: $180 -> $216, c: $95 -> $114, h: $40 -> $48 (all x1.2).

**Predict first:** Cu and Co both change in dollar terms -- does the
critical ratio move? Does Q*?

**What to do:** play 30 rounds, then run the analysis cell. This is the
scenario most people get wrong on the first guess.


In [ ]:
inflation_engine = NewsvendorEngine(p=216, c=114, h=48, mu=300, sigma=60, name="Inflation")
box3, STATE3 = build_single_product_widget(inflation_engine, rounds=30, order_min=0, order_max=500,
                                            order_step=5, session_tag="inflation", title="Chapter 3: Inflation (proportional shock)")
display(box3)


In [ ]:
if "inflation" in SESSION_LOG and "baseline" in SESSION_LOG:
    b, s = SESSION_LOG["baseline"], SESSION_LOG["inflation"]
    compare_bar(["Baseline", "Inflation (x1.2)"], [b["q_star"], s["q_star"]], [b["avg_order"], s["avg_order"]],
                "Chapter 3: Cu and Co both rose -- did Q* actually move?")
    print(f"Cu -- baseline: {b['p']-b['c']:.1f}  ->  inflation: {s['p']-s['c']:.1f}  (went up in dollars)")
    print(f"Co -- baseline: {b['c']-b['h']:.1f}  ->  inflation: {s['c']-s['h']:.1f}  (went up in dollars)")
    print(f"Critical ratio -- baseline: {b['critical_ratio']:.4f}  ->  inflation: {s['critical_ratio']:.4f}")
    print(f"Q* -- baseline: {b['q_star']:.1f}  ->  inflation: {s['q_star']:.1f}  ({s['q_star']-b['q_star']:+.1f})")
    print()
    print("Takeaway: Cu and Co both moved, but their RATIO didn't -- so Q* barely moved either,")
    print("even though it may have felt like 'everything changed.'")
else:
    print("Play both Chapter 0 and Chapter 3 fully before running this cell.")


**Questions to answer -- Chapter 3 (inflation / invariance):**
1. Calculate the new Cu and Co in dollar terms. Did they change from baseline?
2. Calculate the new critical ratio to 4 decimal places. Did it change from
   baseline (0.6071)?
3. Did the formula's Q* change from baseline? By how much?
4. Did your own average order change from baseline? If yes, why do you
   think you reacted to a shock that shouldn't have mathematically mattered?


## Chapter 4 -- Demand Spike: A Blogger Post Goes Viral

*"A fashion blogger's post about the shoe goes mildly viral. More people
want it -- but how predictable that demand is hasn't changed."*

mu: 300 -> 450, sigma stays at 60. Price/cost/salvage back to baseline.

**What to do:** play 30 rounds, then compare to Chapter 0.


In [ ]:
mean_shock_engine = NewsvendorEngine(p=180, c=95, h=40, mu=450, sigma=60, name="Mean Shock")
box4, STATE4 = build_single_product_widget(mean_shock_engine, rounds=30, order_min=0, order_max=700,
                                            order_step=10, session_tag="mean_shock", title="Chapter 4: Demand Spike (mean only)")
display(box4)


In [ ]:
if "mean_shock" in SESSION_LOG and "baseline" in SESSION_LOG:
    b, s = SESSION_LOG["baseline"], SESSION_LOG["mean_shock"]
    compare_bar(["Baseline", "Mean shock"], [b["q_star"], s["q_star"]], [b["avg_order"], s["avg_order"]],
                "Chapter 4: Higher mu -> Q* shifts by roughly the same amount")
    print(f"mu -- baseline: {b['mu']}  ->  shock: {s['mu']}   (sigma unchanged: {b['sigma']})")
    print(f"Q* -- baseline: {b['q_star']:.1f}  ->  shock: {s['q_star']:.1f}  ({s['q_star']-b['q_star']:+.1f})")
else:
    print("Play both Chapter 0 and Chapter 4 fully before running this cell.")


**Questions to answer -- Chapter 4 (demand mean shock):**
1. What is the new mu, and what is sigma?
2. What is the new critical ratio? How does it compare to baseline?
3. By how much did Q* shift compared to baseline? Is that shift close to the
   change in mu (300 -> 450)?
4. What was your average order in this chapter, and how does delta-q here
   compare to delta-q in Chapter 0?


## Chapter 5 -- Uncertainty Shock: An Unfamiliar Market

*"You're launching in a brand-new city with zero sales history. Expected
demand is similar to baseline, but you're far less sure."*

mu stays at 300, sigma: 60 -> 140.

**What to do:** play 30 rounds, then compare to Chapter 0.


In [ ]:
std_shock_engine = NewsvendorEngine(p=180, c=95, h=40, mu=300, sigma=140, name="Std Dev Shock")
box5, STATE5 = build_single_product_widget(std_shock_engine, rounds=30, order_min=0, order_max=800,
                                            order_step=10, session_tag="std_shock", title="Chapter 5: Uncertainty Shock (sigma only)")
display(box5)


In [ ]:
if "std_shock" in SESSION_LOG and "baseline" in SESSION_LOG:
    b, s = SESSION_LOG["baseline"], SESSION_LOG["std_shock"]
    compare_bar(["Baseline", "Sigma shock"], [b["q_star"], s["q_star"]], [b["avg_order"], s["avg_order"]],
                "Chapter 5: Same mean, more uncertainty")
    print(f"sigma -- baseline: {b['sigma']}  ->  shock: {s['sigma']}   (mu unchanged: {b['mu']})")
    print(f"Q* -- baseline: {b['q_star']:.1f}  ->  shock: {s['q_star']:.1f}  ({s['q_star']-b['q_star']:+.1f})")
    print()
    print("Compare this delta to Chapter 4's delta -- which lever moved Q* more,")
    print("a change in the average or a change in the uncertainty?")
else:
    print("Play both Chapter 0 and Chapter 5 fully before running this cell.")


**Questions to answer -- Chapter 5 (uncertainty shock):**
1. What is the new sigma, and what is mu?
2. What is the critical ratio? (It should be identical to baseline -- confirm this.)
3. By how much did Q* rise compared to baseline, even though the critical
   ratio didn't change?
4. Compare the size of the Q* shift in Chapter 4 (mean shock, mu 300->450)
   to Chapter 5 (sigma shock, sigma 60->140). Which lever produced a bigger
   change in Q*? Was that also true of your own order?


## Chapter 6 -- No Critical Ratio Given, Just a Service Level

*"Corporate doesn't tell you price, cost, or salvage for this decision --
they just say: 'maintain a 90% in-stock probability for this SKU.'"*

You don't need Cu or Co to solve this. A target **service level (CSL)** IS a
critical ratio by definition: CSL = P(demand <= Q) = critical ratio. So:

Q* = mu + z(CSL) x sigma

using mu=300, sigma=60 (same demand as baseline).

**What to do:** play 30 rounds ordering to the service-level-implied Q*
(or your own judgment), then run the analysis cell.


In [ ]:
TARGET_CSL = 0.90
service_engine = NewsvendorEngine(p=180, c=95, h=40, mu=300, sigma=60, name="Service Level")
qstar_from_csl = service_engine.q_star_from_csl(TARGET_CSL)
print(f"Target service level: {TARGET_CSL:.0%}")
print(f"Implied Q* = mu + z({TARGET_CSL:.0%}) x sigma = {qstar_from_csl:.1f}")

box6, STATE6 = build_single_product_widget(service_engine, rounds=30, order_min=0, order_max=500,
                                            order_step=5, session_tag="service_level",
                                            title="Chapter 6: Service-Level Target (90%)",
                                            record_qstar=qstar_from_csl)
display(box6)


In [ ]:
if "service_level" in SESSION_LOG and "baseline" in SESSION_LOG:
    b, s = SESSION_LOG["baseline"], SESSION_LOG["service_level"]
    compare_bar(["Baseline\n(margin-based)", "Service level\n(90% target)"],
                [b["q_star"], s["q_star"]], [b["avg_order"], s["avg_order"]],
                "Chapter 6: Policy target vs. margin-optimal Q*")
    print(f"Baseline margin-implied critical ratio: {b['critical_ratio']:.3f} ({b['critical_ratio']:.0%})")
    print(f"Corporate's service-level target: 90%")
    print()
    print("Notice: the 90% policy target may ask for a HIGHER or LOWER Q* than the pure")
    print("margin-based critical ratio would justify. Companies sometimes set service levels")
    print("for brand/customer-satisfaction reasons that go beyond per-unit profit math.")
else:
    print("Play both Chapter 0 and Chapter 6 fully before running this cell.")


**Questions to answer -- Chapter 6 (service level):**
1. Write the target CSL as both a percentage and a decimal.
2. What z-score corresponds to that CSL?
3. What is the implied Q* = mu + z(CSL) x sigma? Show your calculation.
4. Compare this Q* to the baseline margin-based Q* (Chapter 0). Is the 90%
   service-level target more or less conservative than the margin-optimal
   order? What does that tell you about the relationship between service
   level and critical ratio?


## Chapter 7 -- Sensitivity Curves: How Continuously Does Q* Respond?

No play needed here -- this is about seeing the *shape* of the relationship,
not just single before/after snapshots like Chapters 1-2.


In [ ]:
base = SESSION_LOG.get("baseline", {"p": 180, "c": 95, "h": 40, "mu": 300, "sigma": 60})
p0, c0, h0, mu0, sigma0 = base["p"], base["c"], base["h"], base["mu"], base["sigma"]

co_range = np.linspace(10, 150, 40)
q_vs_co = [NewsvendorEngine(p=p0, c=c0, h=(c0 - co), mu=mu0, sigma=sigma0).q_star() for co in co_range]

cu_range = np.linspace(10, 200, 40)
q_vs_cu = [NewsvendorEngine(p=(c0 + cu), c=c0, h=h0, mu=mu0, sigma=sigma0).q_star() for cu in cu_range]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(co_range, q_vs_co, color="#C44E52")
axes[0].axvline(c0 - h0, color="gray", linestyle="--", label=f"Baseline Co = {c0-h0:.0f}")
axes[0].set_xlabel("Co (overage cost)"); axes[0].set_ylabel("Q*")
axes[0].set_title("Q* vs. Co (Cu, mu, sigma held fixed)")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(cu_range, q_vs_cu, color="#55A868")
axes[1].axvline(p0 - c0, color="gray", linestyle="--", label=f"Baseline Cu = {p0-c0:.0f}")
axes[1].set_xlabel("Cu (underage cost)"); axes[1].set_ylabel("Q*")
axes[1].set_title("Q* vs. Cu (Co, mu, sigma held fixed)")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("Notice the shape: Q* is monotonically decreasing in Co, and monotonically increasing in Cu --")
print("but neither is a straight line. The formula is most sensitive to Cu/Co near the middle of")
print("the demand distribution, and flattens out at the extremes.")


**Questions to answer -- Chapter 7 (sensitivity curves):**
1. From the Q* vs. Co chart, approximately how much does Q* fall when Co
   doubles from its baseline value ($55 -> $110)?
2. From the Q* vs. Cu chart, approximately how much does Q* rise when Cu
   doubles from its baseline value ($85 -> $170)?
3. Which curve is steeper near the baseline point -- Q* vs. Co, or Q* vs.
   Cu? What does that tell you about which cost estimate you should be more
   careful getting right?


## Chapter 8 -- Behavioral Framing: Does How You're Told Affect What You Order?

Same numbers as baseline (p=$180, c=$95, h=$40, mu=300, sigma=60) -- played
twice, under two different framings of the **identical decision**.

- **Frame A (Gain framing):** "Every pair you sell earns you profit. Order
  well and maximize your take-home earnings."
- **Frame B (Loss framing):** "Every pair you fail to sell is money lost to
  markdown. Every customer you turn away is profit lost forever. Avoid
  these losses."

**What to do:** play Frame A (15 rounds), then Frame B (15 rounds), then
compare.


In [ ]:
frame_a_text = ("<b>Frame A -- Gain framing:</b> Every pair you sell earns you profit. "
                "Order well and maximize your take-home earnings.")
frame_engine = NewsvendorEngine(p=180, c=95, h=40, mu=300, sigma=60, name="Framing")

box8a, STATE8A = build_single_product_widget(frame_engine, rounds=15, order_min=0, order_max=500,
                                              order_step=5, frame_text=frame_a_text,
                                              session_tag="gain_frame", title="Chapter 8A: Gain Frame")
display(box8a)


In [ ]:
frame_b_text = ("<b>Frame B -- Loss framing:</b> Every pair you fail to sell is money lost to markdown. "
                "Every customer you turn away is profit lost forever. Avoid these losses.")

box8b, STATE8B = build_single_product_widget(frame_engine, rounds=15, order_min=0, order_max=500,
                                              order_step=5, frame_text=frame_b_text,
                                              session_tag="loss_frame", title="Chapter 8B: Loss Frame")
display(box8b)


In [ ]:
if "gain_frame" in SESSION_LOG and "loss_frame" in SESSION_LOG:
    avg_gain = SESSION_LOG["gain_frame"]["avg_order"]
    avg_loss = SESSION_LOG["loss_frame"]["avg_order"]
    fig, ax = plt.subplots(figsize=(5, 4))
    bars = ax.bar(["Gain frame", "Loss frame"], [avg_gain, avg_loss], color=["#4C72B0", "#C44E52"])
    ax.axhline(SESSION_LOG["gain_frame"]["q_star"], color="#DD8452", linestyle="--",
               label=f"Formula Q* = {SESSION_LOG['gain_frame']['q_star']:.0f}")
    ax.set_ylabel("Average order quantity")
    ax.set_title("Chapter 8: Same math, different framing")
    for b, v in zip(bars, [avg_gain, avg_loss]):
        ax.text(b.get_x() + b.get_width()/2, v, f"{v:.0f}", ha='center', va='bottom')
    ax.legend()
    plt.tight_layout(); plt.show()
    print(f"Gain frame average order: {avg_gain:.1f}")
    print(f"Loss frame average order: {avg_loss:.1f}")
    print(f"Difference (Loss - Gain): {avg_loss - avg_gain:+.1f}")
else:
    print("Play both Chapter 8A (gain frame) and Chapter 8B (loss frame) fully before running this cell.")


**Questions to answer -- Chapter 8 (framing):**
1. What was your average order under the Gain frame?
2. What was your average order under the Loss frame?
3. What is the difference (Loss minus Gain)?
4. Does the direction of that difference match what Prospect Theory / loss
   aversion would predict? Explain in 1-2 sentences.


## Chapter 9 -- The Whole Story on One Chart

Run after finishing whichever chapters you played. Good candidate for your
opening presentation slide.


In [ ]:
tags_order = ["baseline", "co_shock", "cu_shock", "inflation", "mean_shock", "std_shock",
              "service_level", "gain_frame", "loss_frame"]
labels_map = {
    "baseline": "0. Baseline", "co_shock": "1. Co shock", "cu_shock": "2. Cu shock",
    "inflation": "3. Inflation", "mean_shock": "4. Mean shock", "std_shock": "5. Sigma shock",
    "service_level": "6. Service lvl", "gain_frame": "8a. Gain", "loss_frame": "8b. Loss",
}
present = [t for t in tags_order if t in SESSION_LOG]

if len(present) >= 2:
    labels = [labels_map[t] for t in present]
    q_stars = [SESSION_LOG[t]["q_star"] for t in present]
    avg_orders = [SESSION_LOG[t]["avg_order"] for t in present]
    x = np.arange(len(labels)); width = 0.35
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.bar(x - width/2, q_stars, width, label="Q*", color="#DD8452")
    ax.bar(x + width/2, avg_orders, width, label="Your average order", color="#4C72B0")
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20)
    ax.set_ylabel("Order quantity")
    ax.set_title("Your order vs. Q*, across every scenario you played")
    ax.legend()
    plt.tight_layout(); plt.show()
else:
    print("Play at least two chapters before running this summary.")


**Questions to answer -- Chapter 9 (whole story):**
1. Looking at your full chart, in which scenario was your delta-q (gap from
   Q*) largest?
2. In which scenario was your delta-q smallest -- i.e., where did you track
   the formula best?
3. What's your hypothesis for why you tracked the formula better in some
   scenarios than others?


## Deliverables & In-Class Presentation Guide (8 minutes)

### Required
Submit CSVs for every chapter you played, plus a table: for each chapter,
your average order, mu-hat, sigma-hat, critical ratio, Q*, and delta-q.

### Slide plan -- 8 slides, 8 minutes (~1 minute each)

**Required (4 slides) -- always present these:**
- **Chapter 0** -- baseline delta-q
- **Chapter 4** -- demand-mean shock: did Q* and your order move together?
- **Chapter 6** -- service level: how a policy target compares to the
  margin-based critical ratio
- **Chapter 9** -- the whole-story chart, as your closing slide
- **Choose any 2 more (2 slides)** from Chapters 1, 2, 3, 5, 7, or 8 -- pick
whichever taught you the most or surprised you.

**Optional, time permitting (up to 2 more slides):** add more chapters from
the same list only if you're comfortably inside the 8-minute limit. It's
fine to stop at 6 slides if that's a tighter, cleaner story.

### Grading emphasis
I personally care less about whether your Q* estimate was exactly "correct" and more
about whether you can **explain what each lever did and why**, and whether
you can honestly point out where your own intuition diverged from the
formula.